# Auditoría del modelo Random Forest sobre CICIDS2017

**Autora:** Betzabeth Querales · Máster Data Analytics · VIU · 2026

---

## Por qué este notebook existe

El notebook principal (`01_cicids2017_lambda_nis2.ipynb`) reporta 1,00 de accuracy y 1,00 de macro-F1. Esa métrica es técnicamente correcta sobre el split aleatorio, pero es **engañosa** porque:

1. **CICIDS2017 contiene flujos cuasi-duplicados** dentro de cada sesión de ataque. Un `train_test_split` aleatorio coloca copias casi idénticas en train y test. Documentado en Engelen, Vanhoef & Rimmer (2021) — *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study* (IEEE S&P Workshops).
2. **El feature `Protocol` actúa como atajo determinista**: 100 % de los ataques observados son TCP.
3. **La separación natural entre clases es extrema** (órdenes de magnitud en `Flow Duration` y `Packet Length Mean`).

Este notebook aplica un protocolo de evaluación corregido:

- Deduplicación exacta sobre las features.
- Baselines de referencia: `DummyClassifier` y `LogisticRegression`.
- Random Forest default y con `class_weight='balanced'`.
- Validación cruzada estratificada k=5 sobre macro-F1.
- Matriz de confusión y métricas por clase.

> **El objetivo no es mejorar el número, es reportarlo honestamente.** Si el macro-F1 cae tras la corrección, se publica la caída.

## 1. Imports y configuración

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

print('scikit-learn listo · versión:', __import__('sklearn').__version__)

## 2. Carga del dataset

Misma fuente que el notebook principal. Sube `friday.csv.zip` a la sesión de Colab (icono de carpeta → Upload) o monta tu Drive.

In [ ]:
DATA_PATH = '/content/friday.csv.zip'   # cámbialo si lo tienes en otra ruta

df = pd.read_csv(DATA_PATH)
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
print(f'Filas iniciales : {len(df):,}')
print(f'Columnas        : {df.shape[1]}')
print(df['Label'].value_counts())

## 3. Definición de features

Mismas variables que en el notebook principal para que la comparación sea justa.

In [ ]:
FEATURES = [
    'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Protocol', 'Dst Port', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
]
print(f'{len(FEATURES)} features seleccionadas')

## 4. Deduplicación — el paso crítico

Eliminamos flujos con la misma combinación exacta de las 10 features + `Label`. Son casi siempre capturas múltiples del mismo evento de ataque.

In [ ]:
antes = len(df)
df_dedup = df.drop_duplicates(subset=FEATURES + ['Label']).reset_index(drop=True)
despues = len(df_dedup)

print(f'Antes  : {antes:,}')
print(f'Después: {despues:,}')
print(f'Eliminados: {antes - despues:,} ({100*(antes-despues)/antes:.1f}% del dataset)')
print('\nNueva distribución por clase:')
print(df_dedup['Label'].value_counts())

## 5. Preparación de X / y y split estratificado

In [ ]:
X = df_dedup[FEATURES]
le = LabelEncoder()
y = le.fit_transform(df_dedup['Label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print('Clases:', list(le.classes_))

## 6. Baseline 1 — `DummyClassifier`

Predice siempre la clase mayoritaria. Es la **cota inferior honesta**: cualquier modelo debe superarla con holgura.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

print('=== DummyClassifier (baseline) ===')
print(classification_report(y_test, y_dummy, target_names=le.classes_, digits=4, zero_division=0))
print(f'Macro-F1: {f1_score(y_test, y_dummy, average="macro", zero_division=0):.4f}')

## 7. Baseline 2 — Regresión Logística

Modelo simple, interpretable, sin trucos. Si Random Forest no supera esto por un margen claro, no hay valor añadido.

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=500, n_jobs=-1, class_weight='balanced'))
])
lr.fit(X_train, y_train)
y_lr = lr.predict(X_test)

print('=== LogisticRegression (balanced) ===')
print(classification_report(y_test, y_lr, target_names=le.classes_, digits=4, zero_division=0))
print(f'Macro-F1: {f1_score(y_test, y_lr, average="macro", zero_division=0):.4f}')

## 8. Random Forest — default

Mismo modelo que el notebook principal, pero sobre el dataset deduplicado.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_rf = rf.predict(X_test)

print('=== RandomForest (default) ===')
print(classification_report(y_test, y_rf, target_names=le.classes_, digits=4, zero_division=0))
print(f'Macro-F1: {f1_score(y_test, y_rf, average="macro", zero_division=0):.4f}')

## 9. Random Forest — `class_weight='balanced'`

Compensa el desbalance brutal entre BENIGN (288k) y Botnet (736).

In [ ]:
rf_bal = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                class_weight='balanced')
rf_bal.fit(X_train, y_train)
y_rfb = rf_bal.predict(X_test)

print('=== RandomForest (balanced) ===')
print(classification_report(y_test, y_rfb, target_names=le.classes_, digits=4, zero_division=0))
print(f'Macro-F1: {f1_score(y_test, y_rfb, average="macro", zero_division=0):.4f}')

## 10. Validación cruzada estratificada k=5

Una sola ejecución train/test puede ser afortunada o desafortunada. Con 5 folds medimos la variabilidad del resultado.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
scores = cross_val_score(rf_cv, X, y, cv=skf, scoring='f1_macro', n_jobs=-1)

print('Macro-F1 por fold:', np.round(scores, 4))
print(f'Media ± desviación: {scores.mean():.4f} ± {scores.std():.4f}')

## 11. Matriz de confusión — Random Forest balanced

In [ ]:
cm = confusion_matrix(y_test, y_rfb)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(cmap='Blues', ax=ax, xticks_rotation=30, values_format='d')
ax.set_title('Matriz de Confusión · RF balanced · dataset deduplicado',
             fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Comparativa final

Tabla resumen con las cuatro métricas que importan para publicar en el portafolio.

In [ ]:
from sklearn.metrics import accuracy_score

resultados = pd.DataFrame([
    ['DummyClassifier',       accuracy_score(y_test, y_dummy), f1_score(y_test, y_dummy, average='macro', zero_division=0)],
    ['LogisticRegression',    accuracy_score(y_test, y_lr),    f1_score(y_test, y_lr,    average='macro', zero_division=0)],
    ['RandomForest (default)',accuracy_score(y_test, y_rf),    f1_score(y_test, y_rf,    average='macro', zero_division=0)],
    ['RandomForest (balanced)',accuracy_score(y_test, y_rfb),  f1_score(y_test, y_rfb,   average='macro', zero_division=0)],
], columns=['Modelo', 'Accuracy', 'Macro-F1'])

resultados = resultados.round(4)
display(resultados)

## 13. Conclusiones del informe de auditoría

Completa esta sección **con los números reales que has obtenido arriba** antes de publicar.

### Lectura honesta

- **Volumen de duplicados detectados:** `____` flujos (`____ %` del dataset).
- **Macro-F1 DummyClassifier:** `____` (cota inferior).
- **Macro-F1 LogisticRegression:** `____`.
- **Macro-F1 RandomForest (default):** `____`.
- **Macro-F1 RandomForest (balanced):** `____`.
- **Macro-F1 CV k=5 (media ± std):** `____ ± ____`.

### Interpretación

Si el macro-F1 en CV sigue por encima de 0,95 tras la deduplicación, el modelo **sí** generaliza bien en este dataset. Si cae, lo reportamos tal cual — la transparencia vale más que el número.

### Próximos pasos

- [ ] Evaluar con `GroupKFold` por `Timestamp` o por ventana de captura.
- [ ] Probar XGBoost como alternativa de referencia.
- [ ] Evaluar transferencia sobre CSE-CIC-IDS2018 (dataset distinto).
- [ ] Añadir un modelo no supervisado (Isolation Forest) para detectar zero-days.

---

**Referencias:**

- Engelen, G., Vanhoef, M., & Rimmer, V. (2021). *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study*. IEEE Security & Privacy Workshops.
- Sharafaldin, I., Lashkari, A. H., & Ghorbani, A. A. (2018). *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization*. ICISSP.